# 🧠 Kendi Yapay Zeka Modelimi Nasıl Yaparım?

Bu notebook, sıfırdan kendi yapay zeka modelinizi oluşturmanız için adım adım rehber niteliğindedir.

## 📋 İçerik
1. **Gerekli Kütüphanelerin Yüklenmesi**
2. **Veri Setinin Hazırlanması**  
3. **Veri Ön İşleme**
4. **Modelin Oluşturulması**
5. **Modelin Eğitilmesi**
6. **Modelin Değerlendirilmesi**
7. **Model ile Tahmin Yapma**

## 🎯 Hedefler
- Yapay zeka modelinin temellerini öğrenmek
- Veri işleme adımlarını anlamak
- Sinir ağı mimarisini kavramak
- Model eğitim sürecini deneyimlemek
- Performans değerlendirme yöntemlerini öğrenmek

## 1. 📦 Gerekli Kütüphanelerin Yüklenmesi ve İçe Aktarılması

Yapay zeka modeli oluşturmak için gerekli olan temel kütüphaneleri yükleyeceğiz:

In [ ]:
# Temel matematik ve veri işleme kütüphaneleri
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Makine öğrenmesi kütüphaneleri
from sklearn.datasets import make_classification, load_iris, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Kendi modelimizi import edelim
import sys
import os
sys.path.append('..')  # Üst dizine çık
from src import NeuralNetwork, ModelTrainer, DataProcessor

# Grafik ayarları
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("✅ Tüm kütüphaneler başarıyla yüklendi!")

## 2. 📊 Veri Setinin Hazırlanması

Yapay zeka modeli eğitmek için veri gereklidir. Farklı veri setlerini keşfedelim:

### 2.1 Sentetik Veri Oluşturma

In [ ]:
# Sentetik veri oluşturalım
processor = DataProcessor()

# İki sınıflı sınıflandırma problemi için veri oluştur
X_synthetic, y_synthetic = processor.generate_synthetic_data(
    n_samples=1000,      # 1000 örnek
    n_features=2,        # 2 özellik (görselleştirme için)
    n_classes=2,         # 2 sınıf
    noise=0.1,           # Az gürültü
    random_state=42      # Tekrarlanabilir sonuçlar için
)

print(f"Sentetik veri boyutu: {X_synthetic.shape}")
print(f"Hedef değişken boyutu: {y_synthetic.shape}")
print(f"Sınıf dağılımı: {np.bincount(y_synthetic.flatten())}")

# Veriyi görselleştir
plt.figure(figsize=(10, 6))
colors = ['red', 'blue']
labels = ['Sınıf 0', 'Sınıf 1']

for i in range(2):
    mask = (y_synthetic.flatten() == i)
    plt.scatter(X_synthetic[mask, 0], X_synthetic[mask, 1], 
               c=colors[i], label=labels[i], alpha=0.7)

plt.xlabel('Özellik 1')
plt.ylabel('Özellik 2')
plt.title('Sentetik Veri Dağılımı')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 2.2 Gerçek Veri Setleri

In [ ]:
# Iris veri setini yükle (ünlü çiçek sınıflandırması)
X_iris, y_iris = processor.load_iris_dataset()

print("Iris Veri Seti:")
print(f"Özellikler: {processor.feature_names}")
print(f"Hedef sınıflar: {processor.target_names}")
print(f"Veri boyutu: {X_iris.shape}")

# İlk birkaç örneği göster
iris_df = pd.DataFrame(X_iris, columns=processor.feature_names)
iris_df['Hedef'] = ['Setosa' if y == 1 else 'Non-Setosa' for y in y_iris.flatten()]
print("\nİlk 10 örnek:")
print(iris_df.head(10))

# Özelliklerin dağılımını görselleştir
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
feature_names = processor.feature_names

for i, feature in enumerate(feature_names):
    ax = axes[i//2, i%2]
    
    # Her sınıf için histogram
    for j, class_name in enumerate(processor.target_names):
        mask = (y_iris.flatten() == j)
        ax.hist(X_iris[mask, i], alpha=0.7, label=class_name, bins=20)
    
    ax.set_xlabel(feature)
    ax.set_ylabel('Frekans')
    ax.set_title(f'{feature} Dağılımı')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. 🔧 Veri Ön İşleme

Makine öğrenmesi modellerinin iyi performans gösterebilmesi için veriyi uygun şekilde hazırlamamız gerekir:

### 3.1 Veriyi Eğitim/Validasyon/Test Olarak Ayırma

In [ ]:
# Iris veri seti ile çalışalım
X, y = X_iris, y_iris

# ModelTrainer ile veriyi böl
dummy_model = NeuralNetwork([4, 1])  # Geçici model
trainer = ModelTrainer(dummy_model)

X_train, X_val, X_test, y_train, y_val, y_test = trainer.prepare_data(
    X, y, 
    test_size=0.2,        # %20 test
    validation_size=0.2,  # %20 validation
    random_state=42
)

print("Veri Bölümü:")
print(f"📚 Eğitim seti: {X_train.shape[0]} örnek")
print(f"🔍 Validasyon seti: {X_val.shape[0]} örnek") 
print(f"🧪 Test seti: {X_test.shape[0]} örnek")

# Her setin sınıf dağılımını kontrol et
def print_class_distribution(y, set_name):
    unique, counts = np.unique(y, return_counts=True)
    print(f"\n{set_name} Sınıf Dağılımı:")
    for label, count in zip(unique, counts):
        class_name = "Setosa" if label == 1 else "Non-Setosa"
        percentage = (count / len(y)) * 100
        print(f"  {class_name}: {count} örnek (%{percentage:.1f})")

print_class_distribution(y_train, "🏃‍♂️ Eğitim")
print_class_distribution(y_val, "🔍 Validasyon")
print_class_distribution(y_test, "🧪 Test")

### 3.2 Veri Normalizasyonu

Özellikler farklı ölçeklerde olabilir. Bu durumda normalizasyon yapmamız gerekir:

In [ ]:
# Normalizasyon öncesi verilerin istatistiklerini göster
print("🔍 Normalizasyon Öncesi İstatistikler:")
print("Eğitim Seti:")
print(f"  Ortalama: {np.mean(X_train, axis=0)}")
print(f"  Std Sapma: {np.std(X_train, axis=0)}")
print(f"  Min: {np.min(X_train, axis=0)}")
print(f"  Max: {np.max(X_train, axis=0)}")

# Normalizasyon yap
X_train_norm, X_val_norm, X_test_norm = trainer.normalize_data(X_train, X_val, X_test)

print("\n✨ Normalizasyon Sonrası İstatistikler:")
print("Eğitim Seti:")
print(f"  Ortalama: {np.mean(X_train_norm, axis=0)}")
print(f"  Std Sapma: {np.std(X_train_norm, axis=0)}")
print(f"  Min: {np.min(X_train_norm, axis=0)}")
print(f"  Max: {np.max(X_train_norm, axis=0)}")

# Normalizasyonun etkisini görselleştir
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Öncesi
ax1.boxplot(X_train.T, labels=processor.feature_names)
ax1.set_title('Normalizasyon Öncesi')
ax1.set_ylabel('Değer')
ax1.tick_params(axis='x', rotation=45)

# Sonrası  
ax2.boxplot(X_train_norm.T, labels=processor.feature_names)
ax2.set_title('Normalizasyon Sonrası')
ax2.set_ylabel('Normalize Değer')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("\n💡 Normalizasyon Faydaları:")
print("✅ Tüm özellikler aynı ölçekte (ortalama ≈ 0, std ≈ 1)")
print("✅ Model eğitimi daha hızlı ve kararlı")
print("✅ Büyük değerli özellikler küçükleri baskılamaz")

## 4. 🏗️ Modelin Oluşturulması

Şimdi kendi yapay zeka modelimizi oluşturalım! Sinir ağı mimarisi tasarlayacağız:

### 4.1 Model Mimarisi Seçimi

In [ ]:
# Model mimarisi tasarla
input_size = X_train.shape[1]  # Giriş boyutu (4 özellik)
output_size = 1                # Çıkış boyutu (binary classification)

# Farklı mimariler deneyelim
architectures = {
    "Basit": [input_size, output_size],
    "Tek Gizli": [input_size, 8, output_size], 
    "İki Gizli": [input_size, 10, 6, output_size],
    "Derin": [input_size, 16, 12, 8, output_size]
}

print("🏗️ Mevcut Model Mimarileri:")
print("-" * 40)
for name, arch in architectures.items():
    param_count = sum(arch[i] * arch[i+1] + arch[i+1] for i in range(len(arch)-1))
    print(f"{name:12}: {arch} ({param_count:,} parametre)")

# En uygun mimariyi seç
chosen_architecture = architectures["İki Gizli"]
print(f"\n✅ Seçilen mimari: {chosen_architecture}")

# Modeli oluştur
model = NeuralNetwork(
    layers=chosen_architecture,
    activation='relu',        # ReLU aktivasyon fonksiyonu
    learning_rate=0.01       # Öğrenme oranı
)

print(f"\n🧠 Model Oluşturuldu!")
print(f"📊 Katman sayısı: {len(chosen_architecture)}")
print(f"⚙️ Aktivasyon: ReLU")
print(f"📈 Öğrenme oranı: 0.01")

# Model özeti
from src.utils import create_model_summary
create_model_summary(model, X_train)

## 5. 🏃‍♂️ Modelin Eğitilmesi

Model eğitimi, yapay zekanın öğrenme sürecidir. Verideki örüntüleri keşfedecek:

### 5.1 Eğitim Süreci

In [ ]:
# Trainer'ı güncelle
trainer = ModelTrainer(model)

print("🚀 Model eğitimi başlıyor...")
print("⏱️ Bu işlem biraz zaman alabilir...")

# Eğitim öncesi performans
print("\n📊 Eğitim Öncesi Performans:")
initial_train_acc = model.evaluate(X_train_norm, y_train)
initial_val_acc = model.evaluate(X_val_norm, y_val)
print(f"  Eğitim doğruluğu: {initial_train_acc:.4f}")
print(f"  Validasyon doğruluğu: {initial_val_acc:.4f}")

# Modeli eğit
import time
start_time = time.time()

trainer.train_with_validation(
    X_train_norm, y_train, 
    X_val_norm, y_val,
    epochs=1000,                    # Maksimum epoch sayısı
    early_stopping_patience=50,     # Erken durdurma sabır
    min_delta=1e-4,                # Minimum iyileşme
    verbose=True                    # İlerleme göster
)

end_time = time.time()
training_time = end_time - start_time

print(f"\n✅ Eğitim tamamlandı!")
print(f"⏰ Süre: {training_time:.2f} saniye")

# Eğitim sonrası performans
print("\n📈 Eğitim Sonrası Performans:")
final_train_acc = model.evaluate(X_train_norm, y_train)
final_val_acc = model.evaluate(X_val_norm, y_val)
print(f"  Eğitim doğruluğu: {final_train_acc:.4f} (+{final_train_acc-initial_train_acc:.4f})")
print(f"  Validasyon doğruluğu: {final_val_acc:.4f} (+{final_val_acc-initial_val_acc:.4f})")

### 5.2 Eğitim Sürecini Görselleştirme

In [ ]:
# Eğitim geçmişini görselleştir
trainer.plot_training_history()

# Eğitim istatistikleri
history = trainer.train_history
if history['loss']:
    print("📊 Eğitim İstatistikleri:")
    print(f"  Toplam epoch: {len(history['loss'])}")
    print(f"  En düşük validation loss: {min(history['val_loss']):.4f}")
    print(f"  En yüksek validation accuracy: {max(history['val_accuracy']):.4f}")
    
    # En iyi epoch'u bul
    best_epoch = np.argmax(history['val_accuracy']) + 1
    print(f"  En iyi epoch: {best_epoch}")
    
    # Son 10 epoch'un ortalaması
    if len(history['val_accuracy']) >= 10:
        recent_avg = np.mean(history['val_accuracy'][-10:])
        print(f"  Son 10 epoch validation acc ortalaması: {recent_avg:.4f}")

print("\n💡 Eğitim Süreci Yorumu:")
if len(history['loss']) > 0:
    final_train_loss = history['loss'][-1]
    final_val_loss = history['val_loss'][-1]
    
    if final_val_loss > final_train_loss * 1.5:
        print("⚠️ Overfitting (aşırı öğrenme) işareti var")
        print("   Çözüm: Daha az epoch, regularization, daha fazla veri")
    elif abs(final_val_loss - final_train_loss) < 0.01:
        print("✅ Model iyi genelleme yapıyor")
    else:
        print("📈 Model henüz daha öğrenebilir")

## 6. 📊 Modelin Değerlendirilmesi

Eğitilmiş modelimizin gerçek performansını test verisi üzerinde ölçelim:

### 6.1 Test Seti Performansı

In [ ]:
# Test seti üzerinde kapsamlı değerlendirme
results = trainer.evaluate_model(
    X_test_norm, y_test, 
    class_names=['Setosa', 'Non-Setosa']
)

# Detaylı metrikler hesapla
from src.utils import calculate_metrics, print_metrics

y_pred = results['predictions']
y_proba = results['probabilities']

detailed_metrics = calculate_metrics(y_test, y_pred, y_proba)
print_metrics(detailed_metrics)

# Confusion matrix görselleştir
trainer.plot_confusion_matrix(
    results['confusion_matrix'],
    class_names=['Non-Setosa', 'Setosa'],
)

print("\n🎯 Model Performans Özeti:")
print(f"✅ Test Doğruluğu: {results['accuracy']:.1%}")
print(f"📊 Precision: {detailed_metrics['precision']:.3f}")
print(f"📈 Recall: {detailed_metrics['recall']:.3f}")
print(f"⚖️ F1-Score: {detailed_metrics['f1_score']:.3f}")

# Performans yorumu
if results['accuracy'] > 0.95:
    print("\n🌟 Mükemmel performans!")
elif results['accuracy'] > 0.90:
    print("\n🎉 Çok iyi performans!")
elif results['accuracy'] > 0.80:
    print("\n👍 İyi performans!")
elif results['accuracy'] > 0.70:
    print("\n⚡ Kabul edilebilir performans")
else:
    print("\n🔧 Model iyileştirme gerekli")

## 7. 🔮 Model ile Tahmin Yapma

Eğitilmiş modelimizi kullanarak yeni veriler üzerinde tahmin yapalım:

### 7.1 Yeni Örnekler Üzerinde Tahmin

In [ ]:
# Yeni çiçek örnekleri oluştur (gerçek değerler)
new_flowers = np.array([
    [5.1, 3.5, 1.4, 0.2],  # Tipik Setosa
    [6.5, 3.0, 5.2, 2.0],  # Tipik Non-Setosa  
    [5.0, 3.0, 1.6, 0.2],  # Belirsiz örnek
    [7.0, 3.2, 4.7, 1.4],  # Orta örnek
])

flower_descriptions = [
    "Küçük ve zarif çiçek",
    "Büyük ve gösterişli çiçek", 
    "Orta boyut çiçek",
    "Büyük orta tip çiçek"
]

print("🌸 Yeni Çiçek Örnekleri:")
print("=" * 60)

# Örnekleri normalize et (eğitim setinden öğrenilen parametrelerle)
new_flowers_norm = trainer.scaler.transform(new_flowers)

for i, (flower, desc) in enumerate(zip(new_flowers, flower_descriptions)):
    # Ham veriyi göster
    print(f"\n🌺 Örnek {i+1}: {desc}")
    print(f"   Sepal uzunluk: {flower[0]:.1f} cm")
    print(f"   Sepal genişlik: {flower[1]:.1f} cm")  
    print(f"   Petal uzunluk: {flower[2]:.1f} cm")
    print(f"   Petal genişlik: {flower[3]:.1f} cm")
    
    # Tahmin yap
    probability = model.predict_proba(new_flowers_norm[i:i+1])[0][0]
    prediction = 1 if probability > 0.5 else 0
    
    # Sonucu göster
    confidence = probability if prediction == 1 else (1 - probability)
    prediction_text = "Setosa" if prediction == 1 else "Non-Setosa"
    
    print(f"   🤖 Tahmin: {prediction_text}")
    print(f"   📊 Olasılık: {probability:.3f}")
    print(f"   🎯 Güven: %{confidence*100:.1f}")
    
    # Güven seviyesi yorumu
    if confidence > 0.9:
        print(f"   💪 Çok yüksek güven!")
    elif confidence > 0.8:
        print(f"   👍 Yüksek güven")
    elif confidence > 0.7:
        print(f"   ⚡ Orta güven") 
    else:
        print(f"   🤔 Düşük güven - belirsiz")

print("\n" + "=" * 60)

### 7.2 Modeli Kaydetme

In [ ]:
# Eğitilmiş modeli kaydet
import os
from datetime import datetime

# Kayıt dizinini oluştur
save_dir = "../models/saved_models"
os.makedirs(save_dir, exist_ok=True)

# Model dosya adı
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_filename = f"iris_model_{timestamp}.json"
model_path = os.path.join(save_dir, model_filename)

# Modeli kaydet
model.save_model(model_path)

# Eğitim raporunu kaydet
report_path, _ = trainer.save_training_report(save_dir, results)

print(f"✅ Model başarıyla kaydedildi!")
print(f"📁 Model dosyası: {model_path}")
print(f"📋 Eğitim raporu: {report_path}")

## 🎉 Tebrikler! Kendi Yapay Zeka Modelinizi Oluşturdunuz!

### 📋 Neler Başardık?

1. ✅ **Veri Hazırlığı**: Iris veri setini yükleyip analiz ettik
2. ✅ **Ön İşleme**: Veriyi normalize ettik ve train/val/test'e böldük  
3. ✅ **Model Tasarımı**: Sinir ağı mimarisi tasarladık
4. ✅ **Eğitim**: Modeli backpropagation ile eğittik
5. ✅ **Değerlendirme**: Performansı ölçtük ve görselleştirdik
6. ✅ **Tahmin**: Yeni veriler üzerinde tahmin yaptık
7. ✅ **Kaydetme**: Modeli gelecekte kullanmak üzere kaydettik

### 🧠 Öğrendiklerimiz

- **Sinir Ağları**: Çok katmanlı perceptron yapısı
- **Backpropagation**: Gradyan tabanlı öğrenme algoritması  
- **Aktivasyon Fonksiyonları**: ReLU, Sigmoid
- **Normalizasyon**: Veri ön işlemenin önemi
- **Overfitting**: Aşırı öğrenme ve early stopping
- **Metrikler**: Accuracy, Precision, Recall, F1-score

### 🚀 Sonraki Adımlar

1. **Farklı Veri Setleri**: MNIST, CIFAR-10 gibi veri setleri deneyin
2. **Model İyileştirme**: Daha derin ağlar, regularization teknikleri
3. **Hiperparametre Optimizasyonu**: Learning rate, batch size tuning
4. **Modern Framework'ler**: TensorFlow, PyTorch öğrenin
5. **Derin Öğrenme**: CNN, RNN, Transformer mimarileri

### 💡 Proje Fikirleri

- 📧 **Email Spam Tespiti**
- 🏠 **Ev Fiyat Tahmini** 
- 🖼️ **Görüntü Sınıflandırması**
- 📈 **Hisse Senedi Analizi**
- 🎬 **Film Öneri Sistemi**

**Artık kendi yapay zeka modellerinizi oluşturabilirsiniz!** 🎯